# Intention Collapse: Experiments v3.0

## Single Source of Truth Architecture

This notebook **only imports from `shared_utils.py`** - no inline code duplication.

### Execution
1. Configure `MODEL_FAMILY` and `BENCHMARK` in Section 2
2. Run all cells
3. Results auto-save to Google Drive
4. Repeat for all 9 combinations

---
## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes
!pip install -q datasets scikit-learn scipy sympy
!pip install -q matplotlib seaborn tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/intention_collapse_v3'
SPLITS_PATH = os.path.join(DRIVE_PATH, 'splits')
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(SPLITS_PATH, exist_ok=True)
print(f"✓ Drive path: {DRIVE_PATH}")

In [ ]:
# Clone repository from GitHub
import os
from pathlib import Path
import shutil

# Remove repo if exists
if Path('intention-collapse-experiments').exists():
    shutil.rmtree('intention-collapse-experiments')

# Clone from GitHub
print("Cloning repository from GitHub...")
!git clone -q https://github.com/patriciomvera/intention-collapse-experiments.git

# Navigate to repository root (no duplicate structure anymore)
os.chdir('intention-collapse-experiments')

# Verify shared_utils.py exists
assert Path('src/shared_utils.py').exists(), "❌ shared_utils.py not found!"
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Repository cloned successfully")
print(f"✓ shared_utils.py ready at: {Path('src/shared_utils.py').absolute()}")

# Add src to Python path
import sys
sys.path.insert(0, 'src')

In [ ]:
# Import everything from shared_utils (SINGLE SOURCE OF TRUTH)
import shared_utils as U

print(f"Loaded shared_utils v{U.CODE_VERSION}")
print(f"Environment: {U.get_environment_info()}")

In [ ]:
# Run sanity checks BEFORE starting experiments
print("Running sanity checks...")
checks_ok = U.run_sanity_checks()

if not checks_ok:
    raise RuntimeError("Sanity checks failed! Fix issues before running experiments.")

# Also run quick unit tests
tests_ok = U.run_unit_tests()
if not tests_ok:
    print("\n⚠️ Some unit tests failed - review before trusting results")

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU required! Enable in Runtime > Change runtime type")

In [ ]:
# Setup HuggingFace token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✓ HF_TOKEN loaded from Colab Secrets")
except:
    import getpass
    HF_TOKEN = getpass.getpass("Enter HuggingFace token: ")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

---
## 2. ⚙️ CONFIGURATION

In [ ]:
# =============================================================================
# ⚙️ CONFIGURATION - CHANGE THESE VALUES FOR EACH RUN
# =============================================================================

MODEL_FAMILY = 'mistral'  # 'mistral', 'llama', or 'qwen'
BENCHMARK = 'gsm8k'       # 'gsm8k', 'math', or 'arc'

# =============================================================================
# Other settings (usually keep default)
# =============================================================================

CONFIG = {
    'subset_size': 200,
    'seed': 42,
    'quantization': '4bit',

    # Generation
    'max_new_tokens_baseline': 50,
    'max_new_tokens_cot': 512,
    'max_new_tokens_babble': 200,  # Shorter for babble

    # Metrics
    'variance_threshold': 0.90,
    'entropy_top_k': 100,

    # Paths
    'drive_path': DRIVE_PATH,
    'splits_path': SPLITS_PATH,
}

# Set all seeds
U.set_all_seeds(CONFIG['seed'])

print("="*60)
print(f"📊 EXPERIMENT: {MODEL_FAMILY.upper()} on {BENCHMARK.upper()}")
print("="*60)
print(f"  Model: {U.MODEL_CONFIGS[MODEL_FAMILY]['name']}")
print(f"  Problems: {CONFIG['subset_size']}")
print(f"  Version: {U.CODE_VERSION}")
print("="*60)

---
## 3. Load Model and Data

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_config = U.MODEL_CONFIGS[MODEL_FAMILY]
print(f"Loading model: {model_config['name']}")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_config['name'], token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_config['name'],
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
    torch_dtype=torch.float16
)
model.eval()

print(f"\n✓ Model loaded!")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Layers: {model.config.num_hidden_layers}")

In [ ]:
# Load dataset with consistent indices (same problems for all models)
print(f"\nLoading {BENCHMARK} dataset...")

# Get or create indices (ensures reproducibility across models)
indices = U.get_or_create_indices(
    benchmark=BENCHMARK,
    subset_size=CONFIG['subset_size'],
    seed=CONFIG['seed'],
    splits_dir=CONFIG['splits_path']
)

# Load problems
problems = U.load_problems(BENCHMARK, indices)

print(f"✓ Loaded {len(problems)} problems")
print(f"  Original indices: {indices[:5]}... (saved for reproducibility)")
print(f"\nExample: {problems[0].question[:150]}...")

---
## 4. Run Experiment

In [ ]:
from tqdm.auto import tqdm
import numpy as np

# Create activation extractor
extractor = U.ActivationExtractor(model, model_config['extraction_layers'])

# Run experiments
conditions = ['baseline', 'enhanced', 'babble']
results = {c: [] for c in conditions}
activations = {c: [] for c in conditions}

max_tokens_map = {
    'baseline': CONFIG['max_new_tokens_baseline'],
    'enhanced': CONFIG['max_new_tokens_cot'],
    'babble': CONFIG['max_new_tokens_babble']
}

print("="*60)
print(f"RUNNING: {MODEL_FAMILY.upper()} on {BENCHMARK.upper()}")
print("="*60)

for condition in conditions:
    print(f"\n--- {condition.upper()} ---")
    max_tokens = max_tokens_map[condition]

    for problem in tqdm(problems, desc=condition):
        result, acts = U.run_single_problem(
            model=model,
            tokenizer=tokenizer,
            extractor=extractor,
            problem=problem,
            condition=condition,
            max_new_tokens=max_tokens,
            entropy_top_k=CONFIG['entropy_top_k']
        )

        results[condition].append(result)
        if acts is not None:
            activations[condition].append(acts)

    # Quick stats
    if condition != 'babble':
        acc = np.mean([r.is_correct for r in results[condition]])
        print(f"  Accuracy: {acc:.1%}")

    ent = np.mean([r.metrics.entropy for r in results[condition]])
    print(f"  Mean entropy: {ent:.3f}")
    print(f"  Stopped by criteria: {np.mean([r.stopped_by_criteria for r in results[condition]]):.1%}")

---
## 5. Compute Metrics

In [ ]:
# Aggregate results
print("\nAggregating results...")

aggregated = {}
for condition in conditions:
    aggregated[condition] = U.aggregate_results(
        results[condition],
        activations[condition],
        variance_threshold=CONFIG['variance_threshold']
    )

    print(f"\n{condition.upper()}:")
    if aggregated[condition].get('accuracy') is not None:
        print(f"  Accuracy: {aggregated[condition]['accuracy']:.1%}")
    print(f"  Entropy: {aggregated[condition]['entropy_mean']:.3f} ± {aggregated[condition]['entropy_std']:.3f}")
    print(f"  dim_eff (global): {aggregated[condition]['dim_eff_global']}")
    print(f"  dim_eff (layers): {aggregated[condition]['dim_eff_layerwise']}")
    print(f"  Tokens generated: {aggregated[condition]['generated_tokens_mean']:.1f}")

In [ ]:
# Train recoverability probes
print("\nTraining recoverability probes...")

probe_results = {}

for condition in ['baseline', 'enhanced']:
    if activations[condition]:
        acts = np.stack(activations[condition], axis=0)
        labels = np.array([r.is_correct for r in results[condition]])

        probe = U.train_recoverability_probe(
            acts, labels,
            cv_folds=5,
            n_bootstrap=1000
        )
        probe_results[condition] = probe

        print(f"\n{condition.upper()}:")
        print(f"  AUROC: {probe.auroc:.3f} [{probe.auroc_ci_low:.3f}, {probe.auroc_ci_high:.3f}]")
        print(f"  Balanced Acc: {probe.balanced_acc:.3f}")
        print(f"  Brier: {probe.brier:.3f}")
        print(f"  Samples: {probe.n_samples}, Pos rate: {probe.pos_rate:.1%}")

---
## 6. Save Results

In [ ]:
# Save everything
print("\nSaving results...")

json_path, npz_path = U.save_experiment_results(
    results=results,
    activations=activations,
    probe_results=probe_results,
    aggregated=aggregated,
    config=CONFIG,
    indices=indices,
    model_family=MODEL_FAMILY,
    benchmark=BENCHMARK,
    drive_path=DRIVE_PATH
)

print(f"\n✓ Results saved:")
print(f"  JSON: {json_path}")
if npz_path:
    print(f"  Activations: {npz_path}")

---
## 7. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
ax = axes[0]
accs = [aggregated[c].get('accuracy', 0) or 0 for c in ['baseline', 'enhanced']]
bars = ax.bar(['Baseline', 'CoT'], accs, color=['steelblue', 'coral'])
ax.set_ylabel('Accuracy')
ax.set_title(f'{MODEL_FAMILY.upper()} on {BENCHMARK.upper()}: Accuracy')
ax.set_ylim(0, 1)
for i, v in enumerate(accs):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center')

# Entropy
ax = axes[1]
ents = [aggregated[c]['entropy_mean'] for c in conditions]
ax.bar(['Baseline', 'CoT', 'Babble'], ents, color=['steelblue', 'coral', 'gray'])
ax.set_ylabel('Entropy (bits)')
ax.set_title('Pre-collapse Intention Entropy')

# Probe AUROC
ax = axes[2]
aurocs = [probe_results.get(c, U.ProbeResults(0.5,0,1,0,0,0,0,0,0)).auroc for c in ['baseline', 'enhanced']]
ci_low = [probe_results.get(c, U.ProbeResults(0.5,0,1,0,0,0,0,0,0)).auroc_ci_low for c in ['baseline', 'enhanced']]
ci_high = [probe_results.get(c, U.ProbeResults(0.5,0,1,0,0,0,0,0,0)).auroc_ci_high for c in ['baseline', 'enhanced']]
yerr = [[a - l for a, l in zip(aurocs, ci_low)], [h - a for a, h in zip(aurocs, ci_high)]]
ax.bar(['Baseline', 'CoT'], aurocs, yerr=yerr, capsize=5, color=['steelblue', 'coral'])
ax.axhline(0.5, color='gray', linestyle='--', label='Chance')
ax.set_ylabel('AUROC')
ax.set_title('Recoverability Probe')
ax.set_ylim(0, 1)

plt.tight_layout()

# Save figure
fig_path = os.path.join(DRIVE_PATH, f"{MODEL_FAMILY}_{BENCHMARK}_summary.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Figure saved: {fig_path}")
plt.show()

---
## 8. Available Results

In [ ]:
# Show all completed experiments
available = U.list_available_results(DRIVE_PATH)

print("\nCompleted experiments:")
print("-" * 40)

# Create completion matrix
models = ['mistral', 'llama', 'qwen']
benchmarks = ['gsm8k', 'math', 'arc']

print(f"{'Model':<12} | {'GSM8K':<8} | {'MATH':<8} | {'ARC':<8}")
print("-" * 45)

for m in models:
    row = [m.upper()]
    for b in benchmarks:
        if (m, b) in available:
            row.append('✓')
        else:
            row.append('○')
    print(f"{row[0]:<12} | {row[1]:<8} | {row[2]:<8} | {row[3]:<8}")

print(f"\nTotal: {len(available)}/9 completed")

---
## Done! 🎉

Next steps:
1. Change `MODEL_FAMILY` and `BENCHMARK` in Section 2
2. Runtime > Restart runtime
3. Run all cells again

After all 9 experiments: run `02_consolidate_results.ipynb`